<a href="https://colab.research.google.com/github/sarah-ahm/Building_Agentic_AI_Systems_Final/blob/main/RepoScribe_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RepoScribe

RepoScribe reads a code package and generates an API reference, user guide, and changelog in which every documented symbol and every citation is verified against a symbol tableextracted with tree-sitter. Anything the model invents is pruned before it is written.

Default target: the **`@lwc/module-resolver`** package from [salesforce/lwc](https://github.com/salesforce/lwc).

## Setup — locate the code and install dependencies

Upload the `reposcribe/` folder to Colab (Files panel → upload) **or** mount Google Drive so this notebook can see the `src/reposcribe/` package, then run this cell.

In [7]:
import os, sys, glob, subprocess

def find_repo_root():
    for c in [".", "reposcribe", "/content/reposcribe", "/content"]:
        if os.path.isfile(os.path.join(c, "src", "reposcribe", "__init__.py")):
            return os.path.abspath(c)
    for hit in glob.glob("/content/**/src/reposcribe/__init__.py", recursive=True):
        return os.path.dirname(os.path.dirname(os.path.dirname(hit)))
    return None

ROOT = find_repo_root()
if ROOT is None:
    raise SystemExit(
        "Could not find the RepoScribe code. Upload the `reposcribe/` folder "
        "(Files panel → upload) or mount Google Drive, then re-run this cell."
    )
os.chdir(ROOT)
if os.path.join(ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, "src"))
print("RepoScribe root:", ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")

RepoScribe root: /content/drive/MyDrive/Colab Notebooks/Building Agentic AI Systems/final/reposcribe
Dependencies installed.


## Step 1 — Tests (offline, deterministic)

The whole pipeline is exercised with the mock LLM. Expect **22 passed**.

In [8]:
!{sys.executable} -m pytest -q

......................                                                   [100%]
22 passed in 1.54s


## Step 2 — Evaluation harness + guardrail ablation (offline)

Scores 14 cases against the ground-truth `SymbolTable` and prints the four metrics, then a fault-injection ablation showing the groundedness guardrail catching a hallucinated symbol and a bogus citation.

In [9]:
!{sys.executable} eval/run_eval.py


RepoScribe evaluation (mock run)
  [PASS] cov-resolveModule            documented in API reference
  [PASS] cov-RegistryType             documented in API reference
  [PASS] cov-RegistryEntry            documented in API reference
  [PASS] cov-ModuleRecord             documented in API reference
  [PASS] cov-AliasModuleRecord        documented in API reference
  [PASS] cov-DirModuleRecord          documented in API reference
  [PASS] cov-NpmModuleRecord          documented in API reference
  [PASS] cov-ModuleResolverConfig     documented in API reference
  [PASS] cite-resolveModule           cited to its real file:line
  [PASS] disc-getModuleEntry          internal helper NOT in public API docs
  [PASS] disc-readJson                private helper NOT in public API docs
  [PASS] disc-resolveModuleFromNpm    internal function NOT in public API docs
  [PASS] grounded-no-hallucination    0 documented symbols missing from the SymbolTable
  [PASS] inject-defense               injected instr

## Step 3 — Generate and render the docs (offline)

Runs the full agentic pipeline on the frozen `@lwc/module-resolver` source and renders the generated API reference inline. `auto_yes=True` skips the human approval gate for the notebook.

In [10]:
from reposcribe.config import Settings
from reposcribe.pipeline import run
from IPython.display import Markdown, display

settings = Settings.from_env(mock=True)
state = run("eval/fixtures/lwc-module-resolver", "out", settings, auto_yes=True, verbose=True)
print("\nmetrics:", state.metrics)

api = next(a for a in state.artifacts if a.kind == "api_reference")
display(Markdown(api.markdown))

[walk] 9 source files, 0 skipped
[symbols] 41 symbols; 8 public
[rag] indexed 41 chunks
[plan] artifacts=['api_reference', 'user_guide', 'changelog']
[route] roles={'internal', 'test', 'public_api', 'types'}; 3 test file(s)
[select] documenting 8 public symbols
[metrics] {'n_files': 9, 'n_symbols': 41, 'n_public': 8, 'coverage': 1.0, 'hallucinated': 0, 'invalid_citations': 0, 'elapsed_sec': 0.08, 'provider': 'mock', 'use_rag': True, 'use_reflection': True}
[write] wrote docs to out

metrics: {'n_files': 9, 'n_symbols': 41, 'n_public': 8, 'coverage': 1.0, 'hallucinated': 0, 'invalid_citations': 0, 'elapsed_sec': 0.08, 'provider': 'mock', 'use_rag': True, 'use_reflection': True}


# API Reference — lwc-module-resolver

## API Reference — src/types.ts

### `AliasModuleRecord` (interface)

```
interface AliasModuleRecord
```

Defined at `src/types.ts:23`.

### `DirModuleRecord` (interface)

```
interface DirModuleRecord
```

Defined at `src/types.ts:28`.

### `ModuleRecord` (type_alias)

```
type ModuleRecord
```

Defined at `src/types.ts:44`.

### `ModuleResolverConfig` (interface)

```
interface ModuleResolverConfig
```

Defined at `src/types.ts:39`.

### `NpmModuleRecord` (interface)

```
interface NpmModuleRecord
```

Defined at `src/types.ts:32`.

### `RegistryEntry` (interface)

```
interface RegistryEntry
```

Defined at `src/types.ts:15`.

### `RegistryType` (constant)

```
RegistryType
```

Defined at `src/types.ts:8`.

## API Reference — src/resolve-module.ts

### `resolveModule` (function)

```
function resolveModule( importee: string, dirname: string, config?: Partial<ModuleResolverConfig> ): RegistryEntry
```

Defined at `src/resolve-module.ts:208`.


## Step 4 (optional) — Live run against real Gemini

Add your key in **Colab → Secrets** (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY`, enable notebook access, then run this cell. It sparse-clones just the target package from the real LWC repo (falling back to the frozen fixture) and documents it with real Gemini output — the guardrails still verify and prune everything against the symbol table.

In [11]:
from reposcribe.config import Settings, get_api_key
from reposcribe.pipeline import run
from IPython.display import Markdown, display

assert get_api_key(), "No GEMINI_API_KEY found. Add it in Colab → Secrets, then re-run."

TARGET = "eval/fixtures/lwc-module-resolver"
if not os.path.isdir("lwc"):
    subprocess.run("git clone --depth 1 --filter=blob:none --sparse "
                   "https://github.com/salesforce/lwc.git", shell=True, check=False)
    subprocess.run("cd lwc && git sparse-checkout set packages/@lwc/module-resolver",
                   shell=True, check=False)
cloned = "lwc/packages/@lwc/module-resolver/src"
if os.path.isdir(cloned):
    TARGET = cloned
print("Documenting:", TARGET)

settings = Settings.from_env(mock=False)
state = run(TARGET, "out_live", settings, auto_yes=True, verbose=True)
print("\nmetrics:", state.metrics)
display(Markdown(next(a for a in state.artifacts if a.kind == "api_reference").markdown))

Documenting: lwc/packages/@lwc/module-resolver/src
[walk] 21 source files, 0 skipped
[symbols] 47 symbols; 8 public
[rag] indexed 47 chunks
[plan] artifacts=['api_reference', 'user_guide', 'changelog']
[route] roles={'internal', 'test', 'public_api', 'types'}; 15 test file(s)
[select] documenting 8 public symbols
[reflect] revising 'resolve-module.ts' section: ['The documentation lacks a formal citation or reference to the source code or specification.', "The return type 'RegistryEntry' is not defined or linked to a type definition."]
[reflect] revising 'types.ts' section: ['Missing citations for all symbols.', 'NpmModuleRecord is referenced in ModuleRecord but lacks a corresponding definition block.']
[metrics] {'n_files': 21, 'n_symbols': 47, 'n_public': 8, 'coverage': 1.0, 'hallucinated': 0, 'invalid_citations': 0, 'elapsed_sec': 18.42, 'provider': 'gemini', 'use_rag': True, 'use_reflection': True}
[write] wrote docs to out_live

metrics: {'n_files': 21, 'n_symbols': 47, 'n_public':

# API Reference — src

### resolveModule

`resolveModule(importee: string, dirname: string, config?: Partial<ModuleResolverConfig>): RegistryEntry`

Resolves LWC modules using a custom resolution algorithm based on the project's `lwc.config.json` or the `lwc` key in `package.json`. The function iterates through configured modules to find a match for the provided specifier.

#### Configuration
For details on available configuration options, refer to the `ModuleResolverConfig` interface definition in the LWC compiler documentation.

#### Return Type
The function returns a `RegistryEntry`, which represents the metadata for the resolved module. See the internal `RegistryEntry` type definition for structure details.

#### Errors
- Throws `LWC_CONFIG_ERROR` if the configuration is invalid.
- Throws `NO_LWC_MODULE_FOUND` if the module cannot be located.

### RegistryType
RegistryType is a constant object defining the supported module registry types.

### RegistryEntry
RegistryEntry defines the structure for a registry entry, containing a `name` (string), `type` (RegistryType), and `path` (string).

### AliasModuleRecord
AliasModuleRecord represents a module alias, containing a `name` (string) and a `path` (string).

### DirModuleRecord
DirModuleRecord represents a directory-based module, containing a `dir` (string).

### NpmModuleRecord
NpmModuleRecord represents an NPM-based module, containing an `npm` (string) and an optional `map` (object with string keys and string values).

### ModuleResolverConfig
ModuleResolverConfig defines the configuration for the module resolver, containing a `rootDir` (string) and a list of `modules` (ModuleRecord[]).

### ModuleRecord
ModuleRecord is a union type representing any valid module record: AliasModuleRecord, DirModuleRecord, or NpmModuleRecord.


---

Generated docs are written to `out/` (offline) and `out_live/` (live): `api_reference.md`, `user_guide.md`, `changelog.md`, plus `workspace_state.json` (the episodic-memory run trace). See `docs/architecture.md` for the design and `eval/eval_report.md` for the full evaluation.